# 04 — The Lua DSL and Forth

Substitute for the discharged DSL guide. `hllset-dsl` wraps the algebra in a
Lua VM (`hllset.*` bindings, operators on `LatticeElement`), and
`hllset-forth` lowers a postfix Forth frontend to the same runtime.


In [2]:
:dep hllset-dsl = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-dsl" }
:dep hllset-forth = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-forth" }


In [3]:
use hllset_dsl::DslRuntime;
use hllset_forth::{compile_to_lua, parse};
let rt: DslRuntime = DslRuntime::new().unwrap();
println!("Lua DSL + Forth loaded");


Lua DSL + Forth loaded


---
## Lua: inscribe and cardinality

`hllset.inscribe({...})` builds a `LatticeElement`; `#` is cardinality.


In [4]:
let n: f64 = rt.eval(r#"
    local a = hllset.inscribe({"hello", "world", "lua"})
    return #a
"#).unwrap();
println!("inscribe 3 tokens -> cardinality {:.1}", n);


inscribe 3 tokens -> cardinality 3.0


---
## Lua: lattice operators

`+` union, `*` intersection, `-` difference, `==` content equality.


In [5]:
let (u, i, d): (f64, f64, f64) = rt.eval(r#"
    local a = hllset.inscribe({"a", "b", "c"})
    local b = hllset.inscribe({"b", "c", "d"})
    return #(a + b), #(a * b), #(a - b)
"#).unwrap();
println!("|A∪B|={:.0}  |A∩B|={:.0}  |A\\B|={:.0}", u, i, d);


|A∪B|=4  |A∩B|=2  |A\B|=1


---
## Lua: tokenize, store, load

The runtime carries an in-memory store; keys are the content addresses.


In [6]:
let ok: bool = rt.eval(r#"
    local e = hllset.tokenize("alpha beta gamma")
    hllset.store(e)
    local keys = hllset.list("h:")
    local back = hllset.load(keys[1])
    return back ~= nil and #back == #e
"#).unwrap();
println!("store -> load roundtrip preserved cardinality: {}", ok);


store -> load roundtrip preserved cardinality: true


---
## Forth: parse, lower, run

Forth source → AST → Lua → the same runtime. One source, many targets.


In [7]:
let forth = r#"
"red" "car" "intersection" 3 INSCRIBE
"slow" "down" "intersection" 3 INSCRIBE
INTERSECT CARD
"#;
let ast = parse(forth).unwrap();
let lua = compile_to_lua(&ast);
println!("compiled Lua:\n{}", lua);
let result: Vec<f64> = rt.eval(&lua).unwrap();
println!("Forth result (intersection cardinality): {:.1}", result[0]);


compiled Lua:
-- Generated by hllset-forth
v0 = "red"
v1 = "car"
v2 = "intersection"
v3 = 3
v4 = hllset.inscribe({ v0, v1, v2 })
v5 = "slow"
v6 = "down"
v7 = "intersection"
v8 = 3
v9 = hllset.inscribe({ v5, v6, v7 })
v10 = v4 * v9
v11 = #v10
return {v11}

Forth result (intersection cardinality): 1.0
